In [ ]:
#@title 1. Environment Setup & Dependency Installation

print("Installing dependencies...")
!pip install sbol3 xmltodict lxml jsonschema pandas matplotlib seaborn ipywidgets gitpython --quiet
print("Dependencies installed.")

print("\nCloning the SimBOL repository...")
# Remove the repository if it already exists to ensure a fresh clone
!rm -rf SimBOL
!git clone https://github.com/DRAGGON-Lab/SimBOL/tree/Colin_summer_research
print("Repository cloned successfully.")

import sys
sys.path.append('/content/SimBOL')
print("Project path added to system.")

# General imports
import json
from google.colab import files
from IPython.display import display, clear_output, HTML

# Specific imports from the SimBOL project modules
try:
    from sbol3_to_json_converter.process_sbol import process_sbol
    from sbol3_to_json_converter.upload_sbol_file import upload_sbol_file
    from json_to_cellmodeller_generator.ui_params import display_form
    from json_to_cellmodeller_generator.params import prepare_parameters_and_data
    from json_to_cellmodeller_generator.cellmodeller_converter import generate_script
    print("SimBOL project modules loaded successfully.")
except ImportError as e:
    print(f"Error importing project modules: {e}")
    print("Please ensure the file structure in GitHub and __init__.py files are correct.")

display(HTML("<hr><b><font color='green'>Setup complete</font></b> You can proceed to the next step."))


Installing dependencies...
Dependencies installed.

Cloning the SimBOL repository...
Cloning into 'Colin_summer_research'...
fatal: repository 'https://github.com/DRAGGON-Lab/SimBOL/tree/Colin_summer_research/' not found
Repository cloned successfully.
Project path added to system.
Error importing project modules: No module named 'sbol3_to_json_converter'
Please ensure the file structure in GitHub and __init__.py files are correct.


In [ ]:
#@title 2. Upload SBOL3 File

sbol_file_path = None
sbol_file_name_only = None

try:
    sbol_file_path, sbol_file_name_only = upload_sbol_file()
    if sbol_file_path:
        display(HTML(f"<hr><b><font color='green'>File '{sbol_file_name_only}' uploaded and saved as '{sbol_file_path}'</font></b>"))
    else:
        display(HTML(f"<hr><b><font color='red'>File upload cancelled or failed.</font></b>"))
except Exception as e:
    print(f"An error occurred during file upload: {e}")
    display(HTML(f"<hr><b><font color='red'>Error during file upload.</font></b>"))

In [ ]:
#@title 3. Convert SBOL3 to JSON

sbol_data_json_internal = None

if 'sbol_file_path' in locals() and sbol_file_path and 'sbol_file_name_only' in locals() and sbol_file_name_only:
    try:
        print(f"Processing '{sbol_file_name_only}'...")
        output_clean_json_path = process_sbol(sbol_file_path, sbol_file_name_only)

        if output_clean_json_path:
            print(f"SBOL file processed. Summarized JSON generated: {output_clean_json_path}")
            # Load the JSON for use in subsequent steps
            with open(output_clean_json_path, "r") as f:
                sbol_data_json_internal = json.load(f)
            display(HTML(f"<hr><b><font color='green'>SBOL file processed. Summarized JSON generated and loaded.</font></b>"))
        else:
            print("The process_sbol function did not return a valid JSON file path.")
            display(HTML(f"<hr><b><font color='red'>Error generating the JSON file.</font></b>"))

    except Exception as e:
        print(f"An error occurred during SBOL file processing: {e}")
        display(HTML(f"<hr><b><font color='red'>Error processing the SBOL file.</font></b>"))
else:
    print("No SBOL3 file was uploaded in the previous step. Please run the upload cell first.")
    display(HTML(f"<hr><b><font color='orange'>Warning: No SBOL file found to process.</font></b>"))

In [ ]:
#@title 4. Display CellModeller Parameter Form

cm_simulation_parameters = None
extracted_genes_data = None
extracted_qs_actions = None
extracted_biochemical_reactions = None

if 'sbol_data_json_internal' in locals() and sbol_data_json_internal:
    try:
        cm_simulation_parameters, extracted_genes_data, extracted_qs_actions, extracted_biochemical_reactions = display_form(sbol_data_json_internal)
        # Save confirmation is handled within display_form
    except Exception as e:
        print(f"An error occurred while displaying the form: {e}")
        display(HTML(f"<hr><b><font color='red'>Error displaying the parameter form.</font></b>"))
else:
    print("SBOL JSON data (sbol_data_json_internal) has not been loaded.")
    print("Please ensure you have successfully run the previous steps (SBOL Upload and Processing).")
    display(HTML(f"<hr><b><font color='orange'>Warning: No data available for the form. Execute previous steps.</font></b>"))

In [ ]:
#@title 5. Generate and Download .py File

if 'cm_simulation_parameters' in locals() and cm_simulation_parameters:
    if extracted_genes_data is not None and extracted_qs_actions is not None and extracted_biochemical_reactions is not None:
        try:
            print("Preparing final data for .py file generation...")
            # Use sbol_data_json_internal which is the loaded JSON
            signals, protein_actions = prepare_parameters_and_data(sbol_data_json_internal, cm_simulation_parameters)

            # Determine the output file name
            base_name_for_output = "simulation_output"
            if 'sbol_file_name_only' in locals() and sbol_file_name_only:
                base_name_for_output = sbol_file_name_only.split('.')[0]

            cm_output_file_name = f"{base_name_for_output}.py"

            print(f"Generating file '{cm_output_file_name}'...")
            generate_cm_file(cm_simulation_parameters, extracted_genes_data, extracted_qs_actions, signals, protein_actions, extracted_biochemical_reactions, gro_output_file_name)
            print(f"File '{cm_output_file_name}' generated successfully.")

            files.download(cm_output_file_name)
            display(HTML(f"<hr><b><font color='green'>.py file generated and download initiated</font></b>"))

        except Exception as e:
            print(f"An error occurred during .py file generation or download: {e}")
            display(HTML(f"<hr><b><font color='red'>Error generating/downloading the .py file.</font></b>"))
    else:
        print("Some extracted data (genes, QS, reactions) is missing. Was the form completed correctly?")
        display(HTML(f"<hr><b><font color='orange'>Warning: Missing data to generate .py file.</font></b>"))
else:
    print("Simulation parameters (cm_simulation_parameters) have not been saved.")
    print("Please ensure you have interacted with the form and pressed 'Save Parameters'.")
    display(HTML(f"<hr><b><font color='orange'>Warning: No parameters available to generate .py file.</font></b>"))